# Built-in Tools: Web Search & Web Fetch

Built-in `WebSearch` and `WebFetch` tools let the agent look up current information without any custom tool code.


In [10]:
from claude_agent_sdk import (
    query,  # one-shot function: ask something, get a stream of messages back
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)


async def run_with_tool(prompt: str, tool: list[str]) -> None:
    # prompt: what to ask Claude to do
    # tool:   the single tool name we're allowing for this run,
    #         e.g. "WebSearch" or "WebFetch"
    options = ClaudeAgentOptions(model="haiku", allowed_tools=tool)
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # Print every tool call Claude makes so we can see it
                # actually searching/fetching, instead of guessing from memory.
                if isinstance(block, ToolUseBlock):
                    print(f"[tool call] {block.name}({block.input})")
        elif isinstance(message, ResultMessage):
            print(f"\nResult: {message.result}")

## Web search


In [11]:
# WebSearch: the built-in tool Claude uses to search the web for current info.
# We explicitly ask it to search instead of answering from memory, since its
# training data has a cutoff date and can't know today's latest Python version.
await run_with_tool(
    "Use the web search tool right now to find the latest stable Python version "
    "— don't answer from memory, your training data is out of date.",
    ["WebSearch"],
)

[tool call] ToolSearch({'query': 'select:WebSearch', 'max_results': 1})
[tool call] WebSearch({'query': 'latest stable Python version 2026'})

Result: Based on current web search results, **the latest stable Python version is Python 3.14.7**, which was released on August 5, 2026. This is the most recent stable release as of August 2026.

For reference, Python 3.14.3 was released earlier in February 2026, but the newer 3.14.7 patch version is now the recommended stable release.

Sources:
- [The Latest Version of Python | phoenixNAP KB](https://phoenixnap.com/kb/latest-python-version)
- [Latest Python Version for Windows (2026) | Krishnamohan Productions](https://krishnamohanproductions.com/blog/tech-blog/latest-python-version-windows-2026/)
- [Python 3.14 in 2026: What's New, What's Faster, and When You'll Actually Feel It | Medium](https://medium.com/@gautsoni/python-3-14-in-2026-whats-new-what-s-faster-and-when-you-ll-actually-feel-it-with-code-4b2b5a62c9fd)
- [Which Python Version Sh

## Web fetch + summarize


In [ ]:
# WebFetch: the built-in tool Claude uses to download and read a specific URL,
# then answer questions about its content (here: summarize it).
# await run_with_tool(
#     "Fetch information about handle @yashjainio in platform like youtube, instagram, medium, github, etc and summarize it in one sentence.",
#     ["WebFetch"],
# )

await run_with_tool("Fetch the todays Bangalore temperature", ["WebFetch", "WebSearch"])

[tool call] ToolSearch({'query': 'select:WebSearch', 'max_results': 1})
[tool call] WebSearch({'query': 'Bangalore temperature today August 19 2026'})

Result: Based on today's weather data for Bangalore (August 19, 2026):

**Current Temperature:** 24°C (feels like 22°C)
**High:** 29°C
**Low:** 20°C
**Precipitation Chance:** 17% chance of rain around 10 PM

It appears to be a pleasant monsoon day in Bangalore with moderate temperatures and a low chance of rainfall.

**Sources:**
- [Bangalore Weather Conditions: Temperature | 30 Days Forecast](https://www.aqi.in/weather/in/india/karnataka/bangalore)
- [Bengaluru, Karnataka, India Monthly Weather | AccuWeather](https://www.accuweather.com/en/in/bengaluru/204108/august-weather/204108)
- [Weather Bengaluru in August 2026: Temperature & Climate](https://en.climate-data.org/asia/india/karnataka/bengaluru-4562/t/august-8/)
- [Weather in Bengaluru in August 2026 - Detailed Forecast](https://www.easeweather.com/asia/india/karnataka/bangalore-ur

## Summary

- `WebSearch` and `WebFetch` remove the need to hand-roll HTTP requests for common lookup tasks.
- Both are gated the same way as any other tool — via `allowed_tools` on `ClaudeAgentOptions`.
